# ch05 结论与决策建议

综合分析结果，基于严格决策框架做出业务决策。

In [ ]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import Config
from src.utils.data_loader import DataLoader
from src.utils.output_manager import OutputManager

print("模块加载成功")

## Step 1: 加载并汇总所有分析结果

In [ ]:
config = Config()
loader = DataLoader(config)
output = OutputManager(config)

# 加载清洗后数据
df = loader.load_processed("cleaned_data.csv")

# 基础统计
n_con = len(df[df['group'] == 'con'])
n_exp = len(df[df['group'] == 'exp'])
print(f"对照组样本量: {n_con}")
print(f"实验组样本量: {n_exp}")

## Step 2: 加载前面章节结果

In [ ]:
# 尝试加载 ch03 结果
ch03_path = config.tables_dir / "ch03_hypothesis_test_results.csv"
if ch03_path.exists():
    ch03_results = pd.read_csv(ch03_path).iloc[0]
    print("ch03 结果加载成功:")
    print(f"  p 值: {ch03_results['p_value']:.6f}")
    print(f"  Cohen's h: {ch03_results['cohens_h']:.4f}")
    print(f"  功效: {ch03_results['power']:.4f}")
else:
    print("ch03 结果文件不存在")
    ch03_results = None

## Step 3: 应用决策框架

In [ ]:
if ch03_results is not None:
    p_value = ch03_results['p_value']
    cohens_h = ch03_results['cohens_h']
    power = ch03_results['power']
    
    print("=== 决策条件评估 ===")
    print(f"p < 0.05: {p_value < 0.05} (p = {p_value:.6f})")
    print(f"Cohen's h >= 0.2: {cohens_h >= 0.2} (h = {cohens_h:.4f})")
    print(f"功效 >= 0.8: {power >= 0.8} (power = {power:.4f})")
    print()
    
    # 决策逻辑
    if p_value < 0.05 and cohens_h >= 0.2 and power >= 0.8:
        decision = "✅ 全量上线"
        reason = "统计显著 + 效应有意义 + 功效充足"
        risk = "低风险"
    elif p_value < 0.05 and cohens_h < 0.2:
        decision = "⚠️ 灰度发布"
        reason = "统计显著但效应量过小"
        risk = "中风险"
    elif p_value >= 0.05 and power < 0.8:
        decision = "🔄 需要进一步实验"
        reason = "不显著且功效不足，可能存在假阴性"
        risk = "高风险"
    else:
        decision = "❌ 放弃该版本"
        reason = "不显著且功效充足，确实无效果"
        risk = "低风险"
    
    print(f"=== 决策结论 ===")
    print(f"决策建议: {decision}")
    print(f"决策理由: {reason}")
    print(f"风险等级: {risk}")
else:
    print("无法做出决策，缺少必要数据")

## Step 4: 生成决策报告

In [ ]:
# 生成 Markdown 报告
report_content = f"""# A/B 测试决策报告

## 实验概况

- **实验时间**: 2024-01-01 ~ 2024-01-07（7天）
- **样本量**: 对照组 {n_con:,} / 实验组 {n_exp:,}
- **核心指标**: 点击率（CTR）

## 关键发现

| 指标 | 数值 |
|------|------|
| p 值 | {p_value:.6f} |
| Cohen's h | {cohens_h:.4f} |
| 统计功效 | {power:.4f} |

## 决策建议

**{decision}**

**理由**: {reason}

**风险等级**: {risk}

---

*报告生成时间: 2026-05-08*
"""

# 保存报告
report_path = config.project_root / "outputs" / "ch05_conclusion_recommendation" / "decision_report.md"
report_path.parent.mkdir(parents=True, exist_ok=True)
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_content)

print(f"决策报告已保存: {report_path}")